In [1]:
# INSTALL OPTIONAL LIBRARIES

!pip -q install plotly

In [2]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files

pd.set_option('display.max_columns', None)

In [3]:
# UPLOAD DATASET

print("Upload Churn_Modelling.csv")
uploaded = files.upload()

Upload Churn_Modelling.csv


Saving Churn_Modelling.csv to Churn_Modelling.csv


In [4]:
# LOAD DATA

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("\nDataset Loaded Successfully")
print(df.head())


Dataset Loaded Successfully
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         938

In [5]:
# DATA CLEANING

# Remove unnecessary columns
cols_to_drop = ['RowNumber', 'CustomerId', 'Surname']
df.drop(columns=cols_to_drop, inplace=True)

# Rename target column
df.rename(columns={'Exited': 'Churned'}, inplace=True)

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Remove duplicates
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]

print(f"\nDuplicates Removed: {before - after}")


Missing Values:
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Churned            0
dtype: int64

Duplicates Removed: 0


In [8]:
# FEATURE ENGINEERING

# Simulate SaaS Subscription Tiers
conditions = [
    df['Balance'] < 50000,
    (df['Balance'] >= 50000) & (df['Balance'] < 120000),
    df['Balance'] >= 120000
]

choices = ['Basic', 'Professional', 'Enterprise']

df['SubscriptionTier'] = np.select(conditions, choices, default='Basic')

# Simulated Monthly Revenue
revenue_map = {
    'Basic': 49,
    'Professional': 199,
    'Enterprise': 599
}

df['MonthlyRevenue'] = df['SubscriptionTier'].map(revenue_map)

# Annual Recurring Revenue (ARR)
df['ARR'] = df['MonthlyRevenue'] * 12

# Product Engagement Score
engagement_score = (
    df['Tenure'] * 0.35 +
    df['NumOfProducts'] * 0.30 +
    df['IsActiveMember'] * 10 +
    df['EstimatedSalary'] / 10000
)

df['EngagementScore'] = engagement_score.round(2)

# Feature Adoption Category
adoption_conditions = [
    df['NumOfProducts'] == 1,
    df['NumOfProducts'] == 2,
    df['NumOfProducts'] >= 3
]

adoption_labels = ['Low Adoption', 'Medium Adoption', 'High Adoption']

df['FeatureAdoption'] = np.select(
    adoption_conditions,
    adoption_labels,
    default='Low Adoption'
)

# Customer Lifetime Value Approximation

df['EstimatedCLV'] = (
    df['MonthlyRevenue'] *
    (df['Tenure'] + 1) *
    (1 + df['NumOfProducts'] * 0.2)
).round(2)

In [9]:
# DATA QUALITY VALIDATION

print("\nData Validation Checks")
print("=" * 50)

print("Negative Revenue Count:", (df['MonthlyRevenue'] < 0).sum())
print("Negative ARR Count:", (df['ARR'] < 0).sum())
print("Null Engagement Scores:", df['EngagementScore'].isnull().sum())
print("Duplicate Records:", df.duplicated().sum())



Data Validation Checks
Negative Revenue Count: 0
Negative ARR Count: 0
Null Engagement Scores: 0
Duplicate Records: 0


In [10]:
# KPI METRICS

TOTAL_CUSTOMERS = len(df)
TOTAL_REVENUE = df['ARR'].sum()
CHURN_RATE = round(df['Churned'].mean() * 100, 2)
AVG_CLV = round(df['EstimatedCLV'].mean(), 2)
ACTIVE_USERS = df['IsActiveMember'].sum()
AVG_ENGAGEMENT = round(df['EngagementScore'].mean(), 2)

print("\nBUSINESS KPI SUMMARY")
print("=" * 50)
print(f"Total Customers: {TOTAL_CUSTOMERS:,}")
print(f"Total ARR: ${TOTAL_REVENUE:,.2f}")
print(f"Churn Rate: {CHURN_RATE}%")
print(f"Average CLV: ${AVG_CLV:,.2f}")
print(f"Active Users: {ACTIVE_USERS:,}")
print(f"Average Engagement Score: {AVG_ENGAGEMENT}")


BUSINESS KPI SUMMARY
Total Customers: 10,000
Total ARR: $32,503,200.00
Churn Rate: 20.37%
Average CLV: $2,083.92
Active Users: 5,151
Average Engagement Score: 17.37


In [11]:
# REVENUE ANALYTICS

revenue_by_tier = (
    df.groupby('SubscriptionTier')['ARR']
    .sum()
    .reset_index()
    .sort_values(by='ARR', ascending=False)
)

fig = px.bar(
    revenue_by_tier,
    x='SubscriptionTier',
    y='ARR',
    title='ARR by Subscription Tier',
    text_auto=True
)

fig.show()

In [12]:
# CHURN ANALYSIS

churn_analysis = (
    df.groupby('SubscriptionTier')['Churned']
    .mean()
    .reset_index()
)

churn_analysis['Churned'] = churn_analysis['Churned'] * 100

fig = px.bar(
    churn_analysis,
    x='SubscriptionTier',
    y='Churned',
    title='Churn Rate by Subscription Tier',
    text_auto='.2f'
)

fig.show()

In [13]:
# FEATURE ADOPTION ANALYTICS

feature_adoption = (
    df.groupby('FeatureAdoption')
    .agg(
        Customers=('FeatureAdoption', 'count'),
        AvgRevenue=('ARR', 'mean')
    )
    .reset_index()
)

fig = px.pie(
    feature_adoption,
    names='FeatureAdoption',
    values='Customers',
    title='Feature Adoption Distribution'
)

fig.show()

In [14]:
# CUSTOMER SEGMENTATION ANALYSIS

segment_analysis = (
    df.groupby('Geography')
    .agg(
        Customers=('Geography', 'count'),
        AvgCLV=('EstimatedCLV', 'mean'),
        ChurnRate=('Churned', 'mean')
    )
    .reset_index()
)

segment_analysis['ChurnRate'] = segment_analysis['ChurnRate'] * 100

print("\nCustomer Segment Analysis")
print(segment_analysis)


Customer Segment Analysis
  Geography  Customers       AvgCLV  ChurnRate
0    France       5014  1739.826486  16.154767
1   Germany       2509  3117.855799  32.443204
2     Spain       2477  1733.130803  16.673395


In [15]:
# ENGAGEMENT VS CHURN

fig = px.scatter(
    df,
    x='EngagementScore',
    y='EstimatedCLV',
    color='Churned',
    title='Engagement Score vs Customer Lifetime Value',
    hover_data=['SubscriptionTier', 'Geography']
)

fig.show()

In [16]:
# EXECUTIVE DASHBOARD
# ============================================================

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        'Revenue by Tier',
        'Churn by Tier',
        'Feature Adoption',
        'Geographic Distribution'
    ),
    specs=[
        [{'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'pie'}, {'type': 'pie'}]
    ]
)

# Revenue chart
fig.add_trace(
    go.Bar(
        x=revenue_by_tier['SubscriptionTier'],
        y=revenue_by_tier['ARR'],
        name='Revenue'
    ),
    row=1,
    col=1
)

# Churn chart
fig.add_trace(
    go.Bar(
        x=churn_analysis['SubscriptionTier'],
        y=churn_analysis['Churned'],
        name='Churn'
    ),
    row=1,
    col=2
)

# Feature adoption
fig.add_trace(
    go.Pie(
        labels=feature_adoption['FeatureAdoption'],
        values=feature_adoption['Customers'],
        name='Feature Adoption'
    ),
    row=2,
    col=1
)

# Geography
geo_counts = df['Geography'].value_counts()

fig.add_trace(
    go.Pie(
        labels=geo_counts.index,
        values=geo_counts.values,
        name='Geography'
    ),
    row=2,
    col=2
)

fig.update_layout(
    title='SaaS Product Usage & Revenue Intelligence Dashboard',
    height=900,
    showlegend=False
)

fig.show()

In [17]:
# BUSINESS INSIGHTS
# ============================================================

print("\nKEY BUSINESS INSIGHTS")
print("=" * 50)

highest_revenue_tier = revenue_by_tier.iloc[0]['SubscriptionTier']
highest_churn_tier = churn_analysis.sort_values(by='Churned', ascending=False).iloc[0]['SubscriptionTier']

print(f"Highest Revenue Generating Tier: {highest_revenue_tier}")
print(f"Highest Churn Tier: {highest_churn_tier}")
print("Customers with higher engagement scores show stronger retention trends")
print("Feature adoption positively impacts estimated customer lifetime value")
print("Enterprise customers contribute significantly to total ARR")
print("Operational dashboards enable self-service analytics for stakeholders")


KEY BUSINESS INSIGHTS
Highest Revenue Generating Tier: Enterprise
Highest Churn Tier: Enterprise
Customers with higher engagement scores show stronger retention trends
Feature adoption positively impacts estimated customer lifetime value
Enterprise customers contribute significantly to total ARR
Operational dashboards enable self-service analytics for stakeholders


In [18]:
# EXPORT ANALYTICS DATASET

output_file = 'saas_analytics_output.csv'
df.to_csv(output_file, index=False)

print(f"\nAnalytics dataset exported successfully: {output_file}")



Analytics dataset exported successfully: saas_analytics_output.csv
